# 11 — Weekly research report

Aggregate every artifact under ``artifacts/{RUN_ID}/`` into the final
Markdown + JSON report. Notebooks 03 → 04 must have run; everything else is
optional and shows "_no data_" when missing.

Prerequisites (minimum to produce a useful report):

- ``candidates.parquet`` from notebook **03**
- ``funnel.json`` from notebook **03**
- ``trades.parquet`` from notebook **04**
- ``summary.json`` from notebook **04**
- ``config.json`` from notebook **04**

Optional inputs that enrich sections:

- ``cf_entry.parquet`` (notebook 05)
- ``cf_exit.parquet`` (notebook 06)
- ``signal_fade.parquet`` (notebook 07)
- ``liquidity_buckets.parquet`` (notebook 08)
- ``reconciliation.parquet`` (notebook 09)
- ``optuna_trials.parquet`` + ``optuna_best.json`` (notebook 10)


In [1]:
# Notebook bootstrap cell. Keep this in every bowaka_lab notebook.
from pathlib import Path
import sys

repo_root = Path.cwd()
while repo_root != repo_root.parent and not (repo_root / "research_notebooks").exists():
    repo_root = repo_root.parent

bowaka_project = repo_root / "research_notebooks" / "bowaka_lab"
src_path = bowaka_project / "src"
if src_path.exists() and str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

import bowaka_lab
from bowaka_lab.utils.env import load_project_dotenv

_loaded_env = load_project_dotenv()
print(f"bowaka_lab {bowaka_lab.__version__}")
print(
    f"bowaka_lab bootstrap: .env loaded from {_loaded_env}"
    if _loaded_env
    else "bowaka_lab bootstrap: no .env found (env vars must be set in shell)"
)


bowaka_lab 0.1.0
bowaka_lab bootstrap: .env loaded from E:\tradingsoftware\quants-lab\.env


## Configuration

In [2]:
RUN_ID         = "bt_iex_default"
ARTIFACTS_ROOT = "research_notebooks/bowaka_lab/artifacts"


## Derived paths

In [3]:
from pathlib import Path

import pandas as pd

from bowaka_lab.metrics.trade_metrics import per_trade_metrics
from bowaka_lab.reports.markdown import ReportInputs
from bowaka_lab.reports.weekly_report import generate_weekly_report
from bowaka_lab.utils import (
    ArtifactPaths,
    artifact_exists,
    load_json,
    load_parquet,
)


artifacts_root = Path(ARTIFACTS_ROOT) if Path(ARTIFACTS_ROOT).is_absolute() else (repo_root / ARTIFACTS_ROOT).resolve()
paths = ArtifactPaths.for_run(RUN_ID, artifacts_root)
paths.ensure_dir()
print(f"artifacts: {paths.root}")


artifacts: E:\tradingsoftware\quants-lab\research_notebooks\bowaka_lab\artifacts\bt_iex_default


## Check prerequisites

In [4]:
# Explicit per-artifact existence checks via the typed ArtifactPaths
# properties. Operators see exactly which artifact is missing without
# having to grep the directory tree.
required_files = {
    "candidates":     paths.candidates,
    "funnel":         paths.funnel,
    "trades":         paths.trades,
    "summary":        paths.summary,
    "config":         paths.config,
}
optional_files = {
    "cf_entry":       paths.cf_entry,
    "cf_exit":        paths.cf_exit,
    "signal_fade":    paths.signal_fade,
    "liquidity":      paths.liquidity,
    "reconciliation": paths.reconciliation,
    "optuna_trials":  paths.optuna_trials,
    "optuna_best":    paths.optuna_best,
}

required_status = {name: p.exists() for name, p in required_files.items()}
optional_status = {name: p.exists() for name, p in optional_files.items()}

print("Required artifacts:")
for name, ok in required_status.items():
    print(f"  {name:20s} {'OK' if ok else 'MISSING'}  -> {required_files[name]}")

print()
print("Optional artifacts (sections that render '_no data_' when missing):")
for name, ok in optional_status.items():
    print(f"  {name:20s} {'OK' if ok else 'missing'}  -> {optional_files[name]}")

missing_required = [k for k, v in required_status.items() if not v]
assert not missing_required, (
    f"Missing required artifacts: {missing_required}.\n"
    "Run notebooks 03 and 04 first."
)


Required artifacts:
  candidates           OK  -> E:\tradingsoftware\quants-lab\research_notebooks\bowaka_lab\artifacts\bt_iex_default\candidates.parquet
  funnel               OK  -> E:\tradingsoftware\quants-lab\research_notebooks\bowaka_lab\artifacts\bt_iex_default\funnel.json
  trades               OK  -> E:\tradingsoftware\quants-lab\research_notebooks\bowaka_lab\artifacts\bt_iex_default\trades.parquet
  summary              OK  -> E:\tradingsoftware\quants-lab\research_notebooks\bowaka_lab\artifacts\bt_iex_default\summary.json
  config               OK  -> E:\tradingsoftware\quants-lab\research_notebooks\bowaka_lab\artifacts\bt_iex_default\config.json

Optional artifacts (sections that render '_no data_' when missing):
  cf_entry             OK  -> E:\tradingsoftware\quants-lab\research_notebooks\bowaka_lab\artifacts\bt_iex_default\cf_entry.parquet
  cf_exit              OK  -> E:\tradingsoftware\quants-lab\research_notebooks\bowaka_lab\artifacts\bt_iex_default\cf_exit.parquet
  

## Build ReportInputs

In [5]:
from bowaka_lab.metrics.bucket_analysis import flatten_variant_column
from bowaka_lab.reports.tables import candidate_rank_distribution

funnel  = load_json(paths.funnel)     if required_status["funnel"]  else None
config  = load_json(paths.config)     if required_status["config"]  else {}
summary = load_json(paths.summary)    if required_status["summary"] else {}

candidates_df = load_parquet(paths.candidates) if required_status["candidates"] else pd.DataFrame()
rank_distribution = candidate_rank_distribution(candidates_df) if not candidates_df.empty else None

trades_df = load_parquet(paths.trades) if required_status["trades"] else pd.DataFrame()
if not trades_df.empty:
    stop_pct = float(config.get("exits", {}).get("stop_pct", 0.08))
    trades_for_report = per_trade_metrics(trades_df, stop_pct=stop_pct)
else:
    trades_for_report = trades_df

cf_entry        = load_parquet(paths.cf_entry)        if optional_status["cf_entry"]        else None
cf_exit         = load_parquet(paths.cf_exit)         if optional_status["cf_exit"]         else None
# Pre-flatten the variant JSON column so Sections 10/11 do not depend on a
# freshly-loaded bucket_analysis module inside the kernel.
cf_entry        = flatten_variant_column(cf_entry)
cf_exit         = flatten_variant_column(cf_exit)
signal_fade_df  = load_parquet(paths.signal_fade)     if optional_status["signal_fade"]     else None
liquidity_df    = load_parquet(paths.liquidity)       if optional_status["liquidity"]       else None
reconciliation  = load_parquet(paths.reconciliation)  if optional_status["reconciliation"]  else None
optuna_trials   = load_parquet(paths.optuna_trials)   if optional_status["optuna_trials"]   else None
optuna_best     = load_json(paths.optuna_best)        if optional_status["optuna_best"]     else None

# ReportInputs accepts a single ``counterfactuals`` DataFrame. We concatenate
# the two grids when both exist so the report's Sections 10/11 can render
# from one consolidated frame (the renderer groups by entry_rule or by
# (stop_pct, target_pct) on its own).
cf_frames = [df for df in (cf_entry, cf_exit) if df is not None and not df.empty]
counterfactuals_df = pd.concat(cf_frames, ignore_index=True) if cf_frames else pd.DataFrame()

inputs = ReportInputs(
    run_id=RUN_ID,
    config_hash="sha256:notebook_11",
    data_feed=(config.get("data", {}) or {}).get("feed", "iex"),
    universe_mode=(config.get("universe", {}) or {}).get("mode", "alpaca_current_assets"),
    prefilter_funnel=funnel,
    candidate_rank_distribution=rank_distribution,
    trades=trades_for_report,
    counterfactuals=counterfactuals_df,
    reconciliation=reconciliation,
    signal_fade=signal_fade_df,
    liquidity=liquidity_df,
    optuna_best=optuna_best,
    optuna_trials=optuna_trials,
    known_limitations=[
        "IEX-only feed (exploratory)" if (config.get("data", {}) or {}).get("feed") == "iex" else "",
        "current-universe survivorship-biased" if (config.get("universe", {}) or {}).get("mode") == "alpaca_current_assets" else "",
    ],
    next_actions=[
        "Refine prefilter gates and re-run 03 + 04.",
        "Inspect Section 13 (liquidity buckets) for ADV-tier recommendations.",
        "If 09 reconciliation surfaced implementation_mismatch rows, resolve before the next paper run.",
    ],
)
print("ReportInputs built; sections will render as follows:")
print(f"  candidates:       {0 if candidates_df.empty else len(candidates_df)}")
print(f"  trades:           {0 if trades_for_report.empty else len(trades_for_report)}")
print(f"  counterfactuals:  {0 if counterfactuals_df.empty else len(counterfactuals_df)}")
print(f"  entry_rule cols:  {'entry_rule' in counterfactuals_df.columns}")
print(f"  stop_pct cols:    {'stop_pct' in counterfactuals_df.columns}")
print(f"  signal_fade:      {'present' if signal_fade_df is not None and not signal_fade_df.empty else 'missing'}")
print(f"  liquidity:        {'present' if liquidity_df is not None and not liquidity_df.empty else 'missing'}")
print(f"  reconciliation:   {'present' if reconciliation is not None and not reconciliation.empty else 'missing'}")
print(f"  optuna:           {'present' if optuna_best else 'missing'}")
print(f"  walk_forward:     {'set (optuna ran)' if inputs.has_walk_forward else 'not set'}")


ReportInputs built; sections will render as follows:
  candidates:       1593
  trades:           1361
  counterfactuals:  237357
  entry_rule cols:  True
  stop_pct cols:    True
  signal_fade:      present
  liquidity:        present
  reconciliation:   missing
  optuna:           present
  walk_forward:     set (optuna ran)


## Generate report

In [6]:
res = generate_weekly_report(output_dir=paths.root, inputs=inputs)
print(f"wrote {res.markdown_path}")
print(f"wrote {res.summary_path}")


wrote E:\tradingsoftware\quants-lab\research_notebooks\bowaka_lab\artifacts\bt_iex_default\weekly_report_bt_iex_default.md
wrote E:\tradingsoftware\quants-lab\research_notebooks\bowaka_lab\artifacts\bt_iex_default\weekly_report_bt_iex_default.json


## Preview

In [7]:
try:
    from IPython.display import Markdown, display
    display(Markdown(res.markdown_path.read_text(encoding="utf-8")))
except Exception:
    print(res.markdown_path.read_text(encoding="utf-8"))


# Bowaka Backtest Report

**Run ID:** bt_iex_default  
**Status:** RESEARCH_ONLY  
**Data vendor/feed:** Alpaca / IEX  
**Universe mode:** alpaca_current_assets  
**Config hash:** `sha256:notebook_11`  

> This report is exploratory. It does not establish live-trading readiness.


> This report uses Alpaca IEX data. IEX is a single-exchange feed and is not consolidated SIP data. Results are exploratory and should not be treated as final evidence of live profitability, especially for volume, RVOL, VWAP, spread, quote-age, and liquidity-dependent decisions.


## 1. Run metadata

- Generated at: 2026-05-18T03:26:13.991404+00:00
- Run ID: `bt_iex_default`
- Config hash: `sha256:notebook_11`


## 2. Data source and feed limitations

- Vendor: alpaca
- Feed: iex
- Adjustment: raw
- Limitations: IEX-only exploratory data

## 3. Universe mode and survivorship-bias warning

- Universe: current Alpaca active/tradable assets.
- Bias: **survivorship-biased**. Historical delisted/inactive names may be missing.

## 4. Config hash and dataset hashes

- Config hash: `sha256:notebook_11`
- _no dataset hashes provided_

## 5. Prefilter funnel

| stage                        |   count |
|------------------------------|---------|
| universe_with_features       | 2004484 |
| passed_universe_gates        |  280706 |
| candidates                   |    1593 |
| rejected_by_signal_gates     |  279113 |
| excluded_by_instrument_class |       0 |

## 6. Candidate rank diagnostics

| rank_bucket   |   candidates |   median_signal_strength |
|---------------|--------------|--------------------------|
| 1-50          |           50 |                 24.2706  |
| 51-100        |          223 |                 12.2878  |
| 101-200       |          572 |                  7.84702 |
| 201-500       |          667 |                  5.45358 |
| 501-1000      |           79 |                  4.05675 |
| 1000+         |            2 |                  3.71924 |

## 7. Trade performance summary

|   trade_count |   win_rate |   mean_pnl_pct |   total_pnl |
|---------------|------------|----------------|-------------|
|          1361 |   0.367377 |     -0.0100191 |    -68016.1 |

## 8. Exit reason summary

| exit_reason   |   count |      pct |
|---------------|---------|----------|
| stop_hit      |     389 | 0.285819 |
| time_stop     |     352 | 0.258633 |
| stop_gap      |     345 | 0.25349  |
| target_hit    |     275 | 0.202057 |

## 9. MFE/MAE analysis

| bucket   |   count |   median_pnl_pct |
|----------|---------|------------------|
| loss     |     121 |        -0.08     |
| 0-5%     |     516 |        -0.08     |
| 5-10%    |     278 |        -0.042583 |
| 10-20%   |     432 |         0.15     |
| >=20%    |      14 |         0.15     |

## 10. Entry timing counterfactuals

| entry_rule          |      n |   win_rate |   mean_pnl_pct |   median_pnl_pct |   target_first_rate |   stop_first_rate |
|---------------------|--------|------------|----------------|------------------|---------------------|-------------------|
| fixed_time_0935     |   1565 |   0.409585 |    -0.00841113 |      -0.0118163  |           0.0408946 |          0.227476 |
| fixed_time_0945     | 226925 |   0.394818 |    -0.00629203 |      -0.00702788 |           0.0359414 |          0.253445 |
| fixed_time_1000     |   1565 |   0.442812 |    -0.0055257  |      -0.00719898 |           0.029393  |          0.165495 |
| opening_range_break |   1565 |   0.417252 |    -0.00712655 |      -0.0100602  |           0.0338658 |          0.191693 |
| vwap_reclaim        |   1565 |   0.417252 |    -0.00712655 |      -0.0100602  |           0.0338658 |          0.191693 |

## 11. Exit surface counterfactuals

|   stop_pct |   target_pct |     n |   mean_pnl_pct |   median_pnl_pct |   win_rate |
|------------|--------------|-------|----------------|------------------|------------|
|       0.05 |         0.1  | 14085 |    -0.00600472 |      -0.0136046  |   0.372311 |
|       0.05 |         0.15 | 14085 |    -0.0060599  |      -0.0142138  |   0.367625 |
|       0.05 |         0.2  | 14085 |    -0.00626567 |      -0.0142291  |   0.365495 |
|       0.05 |         0.25 | 14085 |    -0.00618344 |      -0.0142291  |   0.364643 |
|       0.08 |         0.1  | 14085 |    -0.00639817 |      -0.00609015 |   0.40426  |
|       0.08 |         0.15 | 21910 |    -0.00667718 |      -0.00730489 |   0.406618 |
|       0.08 |         0.2  | 14085 |    -0.00669719 |      -0.00633772 |   0.396379 |
|       0.08 |         0.25 | 14085 |    -0.00665006 |      -0.00633772 |   0.39574  |
|       0.1  |         0.1  | 14085 |    -0.00583919 |      -0.00524931 |   0.411928 |
|       0.1  |         0.15 | 14085 |    -0.00609988 |      -0.00555126 |   0.405325 |
|       0.1  |         0.2  | 14085 |    -0.00636472 |      -0.00562074 |   0.402982 |
|       0.1  |         0.25 | 14085 |    -0.0063284  |      -0.00562074 |   0.402343 |
|       0.12 |         0.1  | 14085 |    -0.00598111 |      -0.00505147 |   0.412993 |
|       0.12 |         0.15 | 14085 |    -0.00624202 |      -0.00529575 |   0.40639  |
|       0.12 |         0.2  | 14085 |    -0.00651945 |      -0.00544497 |   0.404047 |
|       0.12 |         0.25 | 14085 |    -0.00648313 |      -0.00552724 |   0.403408 |

## 12. Signal-fade threshold comparison

| bucket   | eval_label   |   trades |   mean_score |
|----------|--------------|----------|--------------|
| none     | after_close  |      341 |     0.645161 |
| none     | rth          |      318 |     0.795597 |
| soft     | after_close  |      213 |     3.93427  |
| soft     | rth          |      210 |     3.97143  |
| hard     | after_close  |      163 |     7.09202  |
| hard     | rth          |      169 |     7.11834  |
| critical | after_close  |      644 |    11.7624   |
| critical | rth          |      664 |    11.8072   |

## 13. Liquidity bucket analysis

| bucket      |   trades |   win_rate |   median_pnl_pct |   stop_gap_rate | bucket_type   |
|-------------|----------|------------|------------------|-----------------|---------------|
| <$200k      |        0 |   0        |    nan           |        0        | adv_bucket    |
| $200k-$1M   |      857 |   0.34189  |     -0.08        |        0.304551 | adv_bucket    |
| $1M-$5M     |      405 |   0.42963  |     -0.0457926   |        0.17037  | adv_bucket    |
| $5M-$25M    |       96 |   0.34375  |     -0.08        |        0.135417 | adv_bucket    |
| >$25M       |        3 |   0        |     -0.0808165   |        0.666667 | adv_bucket    |
| 0-10bps     |        6 |   0.333333 |     -0.000249242 |      nan        | spread_bucket |
| 10-25bps    |        4 |   0.75     |      0.00170439  |      nan        | spread_bucket |
| 25-50bps    |       13 |   0.384615 |     -0.00351371  |      nan        | spread_bucket |
| 50-100bps   |       34 |   0.617647 |      0.00616661  |      nan        | spread_bucket |
| 100-+infbps |     1304 |   0.359663 |     -0.08        |      nan        | spread_bucket |

## 14. Walk-forward optimization (Optuna)

- **Study:** `bowaka_production_bt_iex_default`
- **Trials:** 2500
- **Best objective:** 2.169563
- **Best parameters:**
  - `atr_pct_min` = `0.03051262989343343`
  - `close_location_min` = `0.7340130537157081`
  - `ema_distance_min` = `0.08883008785458922`
  - `ema_slope_min` = `0.03594512696636021`
  - `entry_rule` = `vwap_reclaim`
  - `gap_pct_max` = `0.3`
  - `max_hold_days` = `2`
  - `range_expansion_max` = `3.0`
  - `range_expansion_min` = `1.162515777135367`
  - `rvol_max` = `8.0`
  - `rvol_min` = `1.3844422014336542`
  - `signal_fade_threshold` = `7`
  - `stop_pct` = `0.10434030942439301`
  - `target_pct` = `0.19958025044224442`

**Top trials by objective value:**
|   trial_number |   objective_value |   param_rvol_min |   param_atr_pct_min |   param_range_expansion_min |   param_close_location_min |   param_ema_distance_min |   param_ema_slope_min | param_rvol_max   |   param_range_expansion_max | param_gap_pct_max   | param_entry_rule   |   param_stop_pct |   param_target_pct |   param_max_hold_days | param_signal_fade_threshold   |
|----------------|-------------------|------------------|---------------------|-----------------------------|----------------------------|--------------------------|-----------------------|------------------|-----------------------------|---------------------|--------------------|------------------|--------------------|-----------------------|-------------------------------|
|           2454 |           2.16956 |          1.22176 |           0.0402558 |                     1.08314 |                   0.814574 |                0.0980149 |             0.0350971 | 8.0              |                           3 | None                | vwap_reclaim       |        0.10404   |           0.24287  |                     3 | None                          |
|           2443 |           2.16956 |          1.30391 |           0.0588927 |                     1.04893 |                   0.788074 |                0.0749592 |             0.0271282 | None             |                           5 | 0.3                 | vwap_reclaim       |        0.0504106 |           0.24287  |                     1 | 9                             |
|            817 |           2.16956 |          1.28496 |           0.0329354 |                     1.18548 |                   0.543952 |                0.0757887 |             0.0305812 | 12.0             |                           3 | 0.3                 | vwap_reclaim       |        0.0709382 |           0.234178 |                     1 | 8                             |
|           2437 |           2.16956 |          1.211   |           0.0483695 |                     1.02897 |                   0.694669 |                0.0659629 |             0.0402912 | None             |                           5 | 0.4                 | fixed_time_0935    |        0.0614401 |           0.248309 |                     1 | None                          |
|           2472 |           2.16956 |          1.32312 |           0.0333326 |                     1.14859 |                   0.654441 |                0.0600047 |             0.0416579 | 5.0              |                           3 | None                | vwap_reclaim       |        0.0778328 |           0.23754  |                     2 | 8                             |
|            832 |           2.16956 |          1.24413 |           0.036362  |                     1.189   |                   0.707772 |                0.0686303 |             0.0431607 | 8.0              |                           5 | 0.2                 | fixed_time_0935    |        0.0692483 |           0.181592 |                     1 | None                          |
|           2428 |           2.16956 |          1.2422  |           0.0474707 |                     1.05425 |                   0.766765 |                0.0605228 |            -0.0123065 | 12.0             |                           4 | None                | fixed_time_1000    |        0.106232  |           0.197684 |                     2 | None                          |
|            350 |           2.16956 |          1.22027 |           0.0531763 |                     1.11236 |                   0.634479 |                0.0827161 |             0.0476995 | 12.0             |                           3 | None                | fixed_time_1000    |        0.0776091 |           0.235897 |                     2 | 8                             |
|            337 |           2.16956 |          1.2125  |           0.0347805 |                     1.02968 |                   0.675817 |                0.0745604 |             0.0391862 | 5.0              |                           3 | 0.2                 | fixed_time_1000    |        0.117127  |           0.231144 |                     1 | 8                             |
|           1691 |           2.16956 |          1.22202 |           0.0361131 |                     1.22784 |                   0.722079 |                0.0259476 |             0.0461647 | 5.0              |                           5 | 0.2                 | or_breakout_15m    |        0.11468   |           0.145142 |                     1 | 8                             |

## 15. Paper-vs-backtest reconciliation

_no paper reconciliation provided_

## 16. Known limitations

- IEX-only feed (exploratory)
- current-universe survivorship-biased

## 17. Stop-ship / research-only status

**Research status:** `research-grade exploratory evidence` (not live-trading approved).
Flags:
- `iex_feed_exploratory`
- `current_universe_survivorship_biased`

## 18. Exact next actions

- Refine prefilter gates and re-run 03 + 04.
- Inspect Section 13 (liquidity buckets) for ADV-tier recommendations.
- If 09 reconciliation surfaced implementation_mismatch rows, resolve before the next paper run.

---

**Research status footer:** This run is classified as `research-grade exploratory evidence`. Live-trading approval requires SIP data + point-in-time universe + walk-forward validation per Report §31.


## Research-status footer

In [8]:
# Reprint the research-status footer for visibility.
flags = []
if (config.get("data", {}) or {}).get("feed") == "iex":
    flags.append("iex_feed_exploratory")
if (config.get("universe", {}) or {}).get("mode") == "alpaca_current_assets":
    flags.append("current_universe_survivorship_biased")
if optuna_trials is None or (optuna_trials is not None and optuna_trials.empty):
    flags.append("walk_forward_not_run")
if reconciliation is not None and not reconciliation.empty:
    if "implementation_mismatch" in set(reconciliation.get("classification", pd.Series([], dtype=str))):
        flags.append("paper_implementation_mismatch_unresolved")

print("Research-status flags:")
for f in flags:
    print(f"  - {f}")


Research-status flags:
  - iex_feed_exploratory
  - current_universe_survivorship_biased
